<a href="https://colab.research.google.com/github/uin33273/GoogleColab_practice/blob/main/%E7%AE%97%E5%AE%9A%E5%9F%BA%E6%BA%961_rev2%E3%83%95%E3%82%A9%E3%83%AB%E3%83%80%E3%82%92%E9%81%B8%E6%8A%9E%E3%81%97%E3%81%A61%E3%83%95%E3%82%A1%E3%82%A4%E3%83%AB%E3%81%B8r1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import glob
import os
from google.colab import files
import zipfile
import io

print("全店舗ファイルのZIPまたはCSVファイルをアップロードしてください")

# 1. 複数のCSVファイル（またはZIP）をアップロード
uploaded = files.upload()
if not uploaded:
    print("ファイルがアップロードされませんでした。")
else:
    # 2. 展開用ディレクトリの準備
    target_dir = 'csv_data'
    if os.path.exists(target_dir):
        import shutil
        shutil.rmtree(target_dir)
    os.makedirs(target_dir)

    for file_name in uploaded.keys():
        if file_name.endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(uploaded[file_name]), 'r') as zip_ref:
                zip_ref.extractall(target_dir)
        else:
            with open(os.path.join(target_dir, file_name), 'wb') as f:
                f.write(uploaded[file_name])

    # 3. CSVデータを統合
    combined_data = []
    csv_files = glob.glob(f'{target_dir}/**/*.csv', recursive=True)

    # 数値変換用ヘルパー（空文字やエラーを0にする）
    def to_num(val):
        try:
            return float(str(val).replace(',', '')) if pd.notna(val) and str(val).strip() != '' else 0
        except:
            return 0

    for file in csv_files:
        try:
            # Shift-JIS(cp932)で読み込み
            df = pd.read_csv(file, encoding='cp932')
            if df.empty:
                continue

            # 最終行を取得（total_rowをSeriesとして取得）
            total_row = df.iloc[-1]

            # --- 列名指定によるデータ取得 ---
            # 指定された列名が存在しない場合に備え、.get() または reindex を利用
            j_val  = total_row.get("利用日数", 0)
            s_val  = total_row.get("算定区分１（日数）", 0)
            u_val  = total_row.get("算定区分２（日数）", 0)
            w_val  = total_row.get("算定区分３（日数）", 0)

            # 延長支援加算
            ce_val = to_num(total_row.get("支援費合計内訳(延長支援区分１日数)", 0))
            cg_val = to_num(total_row.get("支援費合計内訳(延長支援区分２日数)", 0))
            ci_val = to_num(total_row.get("支援費合計内訳(延長支援区分３日数)", 0))
            csum_val = ce_val + cg_val + ci_val

            # その他加算
            bg_val = total_row.get("支援費合計内訳(専門的支援実施加算日数)", 0)
            cc_val = total_row.get("支援費合計内訳(送迎加算日数)", 0)
            as_val = total_row.get("支援費合計内訳(欠席時対応加算日数)", 0)

            # フォルダ名とファイル名の整形
            folder_name = os.path.basename(os.path.dirname(file))
            clean_filename = os.path.basename(file).replace('店', '')

            combined_data.append({
                'Folder': folder_name,
                'filename': clean_filename,
                'J_稼働数': j_val,
                'S_算定区分1': s_val,
                'U_算定区分2': u_val,
                'W_算定区分3': w_val,
                'ce+cg+ci_延長支援': csum_val,
                'BG_専門実施': bg_val,
                'CC_送迎加算': cc_val,
                'AS_欠席加算': as_val
            })
        except Exception as e:
            print(f"エラー（スキップ）: {file} - {e}")

    # 4. 保存とダウンロード
    if combined_data:
        result_df = pd.DataFrame(combined_data)
        output_file = '1-複数を1ファイルに.csv'
        result_df.to_csv(output_file, index=False, encoding='cp932')

        print(f"\n処理完了: {len(combined_data)}件をまとめました。")
        files.download(output_file)
    else:
        print("有効なデータが見つかりませんでした。")

全店舗ファイルのZIPまたはCSVファイルをアップロードしてください


Saving 2025.12.zip to 2025.12 (2).zip

処理完了: 208件をまとめました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>